## Basic MOT 

In [ ]:
"""
MULTI-OBJECT TRACKING PROBLEM:

Frame 1: Detections at positions
  • Detection A: (10, 20)
  • Detection B: (30, 40)
  • Detection C: (50, 60)

Frame 2: New detections
  • Detection 1: (12, 22)  ← Which track is this? A, B, or C?
  • Detection 2: (32, 42)  ← Which track?
  • Detection 3: (52, 62)  ← Which track?

CHALLENGE: Match detections to tracks optimally!
"""

import numpy as np
import matplotlib.pyplot as plt

# Visualize the problem
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Frame 1
tracks_frame1 = np.array([[10, 20], [30, 40], [50, 60]])
ax1.scatter(tracks_frame1[:, 0], tracks_frame1[:, 1], 
            s=200, c=['red', 'blue', 'green'], marker='s')
for i, pos in enumerate(tracks_frame1):
    ax1.text(pos[0]+1, pos[1]+1, f'Track {chr(65+i)}', fontsize=12)
ax1.set_title('Frame 1: Known Tracks', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 70)
ax1.set_ylim(0, 80)
ax1.grid(True, alpha=0.3)

# Frame 2
detections_frame2 = np.array([[12, 22], [32, 42], [52, 62]])
ax2.scatter(detections_frame2[:, 0], detections_frame2[:, 1], 
            s=200, c='orange', marker='o')
for i, pos in enumerate(detections_frame2):
    ax2.text(pos[0]+1, pos[1]+1, f'Detection {i+1}', fontsize=12)
ax2.set_title('Frame 2: New Detections - Which is which?', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 70)
ax2.set_ylim(0, 80)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Distance Calculation

In [ ]:
def euclidean_distance(point1, point2):
    """Calculate distance between two points"""
    return np.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

def compute_cost_matrix(detections, tracks):
    """
    Compute cost matrix for assignment problem
    
    Cost[i,j] = distance from detection i to track j
    """
    n_detections = len(detections)
    n_tracks = len(tracks)
    
    cost_matrix = np.zeros((n_detections, n_tracks))
    
    for i, det in enumerate(detections):
        for j, track in enumerate(tracks):
            cost_matrix[i, j] = euclidean_distance(det, track)
    
    return cost_matrix

# Test on our example
cost_matrix = compute_cost_matrix(detections_frame2, tracks_frame1)

print("\n📊 COST MATRIX (smaller = better match):")
print("         Track A  Track B  Track C")
for i in range(len(detections_frame2)):
    print(f"Det {i+1}:    {cost_matrix[i, 0]:.2f}     {cost_matrix[i, 1]:.2f}     {cost_matrix[i, 2]:.2f}")


## Visualize Association

In [ ]:
from scipy.optimize import linear_sum_assignment

# Our example data
tracks = np.array([[10, 20], [30, 40], [50, 60]])
detections = np.array([[12, 22], [32, 42], [52, 62]])

# Compute cost matrix
cost_matrix = compute_cost_matrix(detections, tracks)

# Solve with Hungarian algorithm
det_indices, track_indices = linear_sum_assignment(cost_matrix)

# Visualize the matches
plt.figure(figsize=(10, 8))

# Plot tracks
plt.scatter(tracks[:, 0], tracks[:, 1], s=300, c='blue', marker='s', 
            label='Tracks (Frame 1)', edgecolor='black', linewidth=2)
for i, pos in enumerate(tracks):
    plt.text(pos[0]-3, pos[1]-3, f'T{i}', fontsize=14, fontweight='bold')

# Plot detections
plt.scatter(detections[:, 0], detections[:, 1], s=300, c='red', marker='o', 
            label='Detections (Frame 2)', edgecolor='black', linewidth=2)
for i, pos in enumerate(detections):
    plt.text(pos[0]+2, pos[1]+2, f'D{i}', fontsize=14, fontweight='bold')

# Draw associations
colors = ['green', 'purple', 'orange']
for (det_idx, track_idx), color in zip(zip(det_indices, track_indices), colors):
    plt.arrow(tracks[track_idx, 0], tracks[track_idx, 1],
             detections[det_idx, 0] - tracks[track_idx, 0],
             detections[det_idx, 1] - tracks[track_idx, 1],
             head_width=2, head_length=1, fc=color, ec=color, linewidth=3,
             length_includes_head=True, alpha=0.7)
    
    # Add cost label
    mid_x = (tracks[track_idx, 0] + detections[det_idx, 0]) / 2
    mid_y = (tracks[track_idx, 1] + detections[det_idx, 1]) / 2
    cost = cost_matrix[det_idx, track_idx]
    plt.text(mid_x, mid_y, f'{cost:.1f}m', fontsize=10, 
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.xlabel('X (meters)', fontsize=12)
plt.ylabel('Y (meters)', fontsize=12)
plt.title('Hungarian Algorithm: Optimal Detection-to-Track Assignment', 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

